# 10 — Tier Sweep (run entirely inside Jupyter)

Use this notebook **instead of** `experiments/run_tier_sweep.py` when an
application-control policy (AppLocker / WinError 4551) blocks the `nbconvert`
executable. It does the same thing the driver did, but with `%run`, which
executes each pipeline notebook **in this same kernel** — no external process,
nothing for AppLocker to block.

**Before running this notebook:**
1. `.env` exists at the repo root with `OPENAI_API_KEY` and the three `LLM_MODEL_*`.
2. You have run `python experiments/patch_notebooks.py` once (matcher tier +
   cache control + nb00 pricing). If you cannot run that either, see the
   *Manual patch* note at the bottom.
3. Start Jupyter from the **repo root** (the folder that contains `notebooks/`,
   `artifacts/`, `.env`), so relative paths resolve.

What it produces: `experiments/results/<tier>/run_<seed>/five_axis_report.json`
for every run, plus aggregated `experiments/results/tier_comparison.{json,csv}`.
Then run notebook `09_tier_sweep_aggregation.ipynb` for the paper tables.

In [ ]:
# --- Configuration --------------------------------------------------------
TIERS        = ["weak", "mid", "strong"]   # tiers to sweep
SEEDS        = [42, 43, 44, 45, 46]         # 5 runs per tier
MATCHER_TIER = "strong"                     # accuracy-axis matcher held fixed
DISABLE_CACHE = True                        # independent repeats (REQUIRED)

# IMPORTANT (re-run hygiene): a previous run of this sweep may have left an
# artifacts/llm_cache with entries that predate the seed-aware cache keys, and
# an experiments/results with stale per-run reports. With DISABLE_CACHE=True
# the pipeline calls now bypass the cache and each seed uses a distinct salt,
# so the five runs are genuinely independent (non-zero std). If you want a
# fully clean slate you MAY delete artifacts/llm_cache/*.json and
# experiments/results/ before running; it is not required, only tidier.

# Pipeline notebooks, in execution order (04 writes five_axis_report.json).
PIPELINE = [
    "01_agent1_task_extraction.ipynb",
    "015_agent1_5_process_graph.ipynb",
    "02_agent2_time_estimation.ipynb",
    "03_agent3_roi_computation.ipynb",
    "04_five_axis_validation.ipynb",
]

In [ ]:
import os, json, shutil, statistics, csv
from pathlib import Path
from datetime import datetime, timezone

# Resolve the repo root whether Jupyter started in repo root or in notebooks/.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
NOTEBOOKS = ROOT / "notebooks"
INFER     = ROOT / "artifacts" / "inference"
MANIFEST  = ROOT / "artifacts" / "run_manifest.json"
RESULTS   = ROOT / "experiments" / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

assert (ROOT / ".env").exists(), f".env not found at {ROOT}. Create it first."
assert MANIFEST.exists(), f"run_manifest.json not found at {MANIFEST}."

# Sanity: is the matcher patch present in nb04? If not, the matcher will follow
# the pipeline tier instead of being fixed — warn but continue.
_nb04 = json.loads((NOTEBOOKS / "04_five_axis_validation.ipynb").read_text(encoding="utf-8"))
_nb04_src = "\n".join("".join(c["source"]) for c in _nb04["cells"] if c["cell_type"]=="code")
if "ROI_MATCHER_TIER" not in _nb04_src:
    print("[WARN] nb04 is NOT patched for a fixed matcher tier. Run\n"
          "       python experiments/patch_notebooks.py   (or see Manual patch note).")
print(f"[OK] repo root: {ROOT}")

In [ ]:
ORIGINAL_MANIFEST = MANIFEST.read_text(encoding="utf-8")  # to restore at the end

def set_run(tier, seed):
    """Point the manifest and env vars at one (tier, seed) before %run."""
    m = json.loads(MANIFEST.read_text(encoding="utf-8"))
    m["default_tier"] = tier
    m["seed"] = seed
    for stage in m.get("pipeline_cfg", {}):
        m["pipeline_cfg"][stage]["tier"] = tier
    MANIFEST.write_text(json.dumps(m, ensure_ascii=False, indent=2), encoding="utf-8")
    os.environ["ROI_ACTIVE_TIER"] = tier
    os.environ["ROI_SEED"] = str(seed)
    os.environ["ROI_MATCHER_TIER"] = MATCHER_TIER
    os.environ["ROI_DISABLE_PIPELINE_CACHE"] = "1" if DISABLE_CACHE else "0"

def save_run(tier, seed):
    dest = RESULTS / tier / f"run_{seed}"
    dest.mkdir(parents=True, exist_ok=True)
    shutil.copy2(INFER / "five_axis_report.json", dest / "five_axis_report.json")
    if (INFER / "agent3_roi.json").exists():
        shutil.copy2(INFER / "agent3_roi.json", dest / "agent3_roi.json")
    return json.loads((dest / "five_axis_report.json").read_text(encoding="utf-8"))

def run_notebook_exec(nb_path):
    """Execute a notebook's code cells in a FRESH namespace via exec().
    No %run, no nbconvert.exe, no file-finder, no quoting -- the path is a
    plain pathlib.Path, so spaces / Korean characters cause no trouble. Each
    pipeline notebook is self-contained (re-imports what it needs), so a fresh
    namespace per notebook is correct and avoids cross-notebook variable clashes.
    """
    nb = json.loads(Path(nb_path).read_text(encoding="utf-8"))
    src = "\n".join("".join(c["source"]) for c in nb["cells"]
                    if c["cell_type"] == "code")
    ns = {"__name__": "__main__", "__file__": str(nb_path)}
    exec(compile(src, str(nb_path), "exec"), ns)

print("[OK] helpers defined")

In [ ]:
# --- The sweep ------------------------------------------------------------
# Each pipeline notebook is executed by run_notebook_exec() (defined above),
# which reads the .ipynb and exec()s its code cells. This sidesteps %run's
# file-finder entirely, so the folder path with spaces / Korean characters is
# no longer a problem. A failed run is isolated and reported, not fatal.

import os
os.chdir(ROOT)  # ensure .env / artifacts resolve for every notebook
print(f"[cwd] {os.getcwd()}")

ACC = {"1_accuracy": ["grade_agreement","grade_cohen_kappa","time_MAE_min",
                      "time_MAPE_pct","matched_pairs","unmatched_reference"],
       "2_reliability": ["mean_cv_minutes","median_cv_minutes","pct_nodes_low_cv"],
       "3_efficiency": ["annual_saving_usd","pipeline_llm_cost_usd","saving_to_cost_ratio"],
       "4_transparency": ["grade_rationale_coverage_pct","pct_estimates_grounded"],
       "5_robustness": ["clarifying_rate_pct","missing_system_flagged"]}

def flatten(rep):
    out = {}
    for ax, keys in ACC.items():
        for k in keys:
            out[f"{ax}.{k}"] = rep["axes"][ax].get(k)
    return out

per_tier = {t: [] for t in TIERS}
failures = []
try:
    for tier in TIERS:
        for seed in SEEDS:
            print(f"\n=== tier={tier} seed={seed} ===", flush=True)
            set_run(tier, seed)
            try:
                for nb_file in PIPELINE:
                    print(f"   run {nb_file}", flush=True)
                    run_notebook_exec(NOTEBOOKS / nb_file)
                rep = save_run(tier, seed)
                per_tier[tier].append(flatten(rep))
                acc = rep["axes"]["1_accuracy"]
                print(f"   -> kappa={acc.get('grade_cohen_kappa')} "
                      f"MAPE={acc.get('time_MAPE_pct')}%", flush=True)
            except Exception as e:
                import traceback
                failures.append((tier, seed, repr(e)))
                print(f"   [SKIPPED tier={tier} seed={seed}] {type(e).__name__}: {e}",
                      flush=True)
                traceback.print_exc()
finally:
    MANIFEST.write_text(ORIGINAL_MANIFEST, encoding="utf-8")
    print("\n[restored original run_manifest.json]")

if failures:
    print(f"\n[WARN] {len(failures)} run(s) failed:")
    for t, s, err in failures:
        print(f"   tier={t} seed={s}: {err[:160]}")
else:
    print("\n[OK] all runs completed and saved.")

In [ ]:
# --- Aggregate to tier_comparison.{json,csv} ------------------------------
def agg(vals):
    nums = [v for v in vals if isinstance(v, (int, float))]
    if not nums:
        return {"mean": None, "std": None, "n": 0}
    return {"mean": round(statistics.mean(nums), 4),
            "std": round(statistics.pstdev(nums), 4) if len(nums) > 1 else 0.0,
            "n": len(nums)}

metric_names = sorted({k for runs in per_tier.values() for r in runs for k in r})
summary = {"generated_utc": datetime.now(timezone.utc).isoformat(),
           "seeds": SEEDS, "matcher_tier": MATCHER_TIER, "tiers": {}}
rows = []
for tier in TIERS:
    summary["tiers"][tier] = {}
    for mn in metric_names:
        a = agg([r.get(mn) for r in per_tier[tier]])
        summary["tiers"][tier][mn] = a
        rows.append({"tier": tier, "metric": mn, "mean": a["mean"],
                     "std": a["std"], "n": a["n"]})

(RESULTS / "tier_comparison.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
with (RESULTS / "tier_comparison.csv").open("w", newline="", encoding="utf-8-sig") as f:
    w = csv.DictWriter(f, fieldnames=["tier","metric","mean","std","n"]); w.writeheader(); w.writerows(rows)

print("Wrote experiments/results/tier_comparison.json and .csv\n")
print("Headline (mean across runs):")
for tier in TIERS:
    t = summary["tiers"][tier]
    print(f"  {tier:6s} kappa={t.get('1_accuracy.grade_cohen_kappa',{}).get('mean')} "
          f"MAPE={t.get('1_accuracy.time_MAPE_pct',{}).get('mean')}%")

## After this notebook

Run `09_tier_sweep_aggregation.ipynb` to produce the paper-ready tables
(`paper_headline_table.{csv,tex}` and `paper_full_table.{csv,tex}`), then send me
`experiments/results/tier_comparison.json`.

## Manual patch note (only if you could not run `patch_notebooks.py`)

If `python experiments/patch_notebooks.py` is also blocked, you can make the two
essential edits by hand:

1. **nb04 matcher tier** — in `04_five_axis_validation.ipynb`, find the line
   `mres = llm_call(MATCH_SYSTEM, MATCH_USER, tier=DEFAULT_TIER, ...)` and change
   `tier=DEFAULT_TIER` to `tier=os.environ.get("ROI_MATCHER_TIER", "strong")`.
2. **cache off for repeats** — in each of `01`, `015`, `02`, `03`, right after the
   cell that defines `llm_call`, add a new cell:
   ```python
   import os as _oc
   if _oc.getenv("ROI_DISABLE_PIPELINE_CACHE", "0") == "1":
       _orig = llm_call
       def llm_call(*a, **k):
           k["use_cache"] = False
           return _orig(*a, **k)
   ```
If you skip these, the sweep still runs, but the matcher will follow each
pipeline tier (less clean comparison) and cached calls may collapse repeats.